# Chunking
Divide extracted text into smaller chunks, each focused on a single topic.

In [60]:
import json
import re
from pathlib import Path
from tqdm import tqdm

# Docling — correct import paths for docling >= 2.x
from docling_core.types.doc import DoclingDocument, DocItemLabel
from docling.chunking import HybridChunker

# Tokenizer
from transformers import AutoTokenizer

DOCTAGS_PATH   = "../data/extracted_text/doctags_progress.txt"
PAGE_DESC_PATH = "../data/extracted_text/page_descriptions.json"
METADATA_PATH  = "../data/page_images/metadata.json"
OUTPUT_PATH    = "../data/chunks/enriched_chunks.json"

MAX_TOKENS      = 200
OVERLAP         = 30
TOKENIZER_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
MAX_FIGURES     = 10     # max figure descriptions injected per page
MIN_CHUNK_CHARS = 100    # discard chunks shorter than this

In [61]:
import transformers
import logging

# Suppress the 512 token warning globally for this tokenizer
transformers.logging.set_verbosity_error()

In [62]:
#Load all files
with open(DOCTAGS_PATH, "r", encoding="utf-8") as f:
    doctags_raw = f.read()

with open(PAGE_DESC_PATH, "r", encoding="utf-8") as f:
    page_descriptions = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Files loaded successfully")

Files loaded successfully


In [63]:
#Split Doctags per page

pages = doctags_raw.split("<<PAGE_BREAK>>")

# Remove empty segments caused by trailing PAGE_BREAK delimiter
pages = [p for p in pages if p.strip()]

print(f"Total pages: {len(pages)}")

Total pages: 654


In [64]:
#Clean doctags to raw text
def extract_text_from_doctags(doctag_page: str) -> str:
    """
    Extract plain text from doctag XML.
    Covers all text-bearing tags: text, section headers, page headers, list items, captions.
    Skips non-text tags: picture, otsl (tables handled separately).
    """
    TEXT_TAGS = [
        "text",
        "section_header_level_1",
        "section_header_level_2",
        "section_header_level_3",
        "page_header",
        "page_footer",
        "list_item",
        "caption",
        "footnote",
    ]

    pattern = "|".join(TEXT_TAGS)
    matches = re.findall(
        rf"<({pattern})[^>]*>(.*?)</\1>",
        doctag_page,
        re.DOTALL
    )

    extracted = []
    for tag, content in matches:
        clean = re.sub(r"<[^>]+>", "", content).strip()

        # Strip standalone page numbers e.g. "1-11", "2-34"
        clean = re.sub(r"^\d+-\d+$", "", clean, flags=re.MULTILINE)

        # Strip document reference codes e.g. "76TS11005", "K15C"
        clean = re.sub(r"\b\d{2}[A-Z]{2}\d{5}\b", "", clean)

        clean = clean.strip()
        if clean:
            extracted.append(clean)

    return "\n".join(extracted)

In [65]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=MAX_TOKENS,
    overlap=OVERLAP
)

print("Chunker ready")

Chunker ready


In [66]:
#Inject Figure descriptions
def inject_figures(text: str, page_num: int):
    """
    Return page text and figure descriptions as SEPARATE segments.
    Applies -1 offset to align page_descriptions.json (0-indexed)
    with doctags (1-indexed).
    """
    key = f"page_{page_num - 1}"   #offset fix
    figures = page_descriptions.get(key, {}).get("figures", [])[:MAX_FIGURES]

    # Each figure description becomes its own protected string
    figure_segments = [f"[FIGURE]: {f}" for f in figures]

    return text, figure_segments, figures

In [67]:
#build docling document per page
def build_doc(page_text: str) -> DoclingDocument:
    """Build a minimal DoclingDocument from plain text."""
    doc = DoclingDocument(name="page")
    doc.add_text(label=DocItemLabel.TEXT, text=page_text)
    return doc

In [68]:
#chunk single page
def extract_heading(text: str):
    """Return the first short all-caps line as a heading, or None."""
    for line in text.split("\n")[:5]:
        line = line.strip()
        if line.isupper() and 1 <= len(line.split()) <= 10:
            return line
    return None


def build_contextualized(chunk, chunk_text: str) -> str:
    """
    Safely build a context-enriched string using chunk metadata headings.
    Falls back to plain chunk_text if metadata is unavailable.
    """
    try:
        headings = chunk.meta.headings or []
        if headings:
            return " > ".join(headings) + "\n" + chunk_text
    except AttributeError:
        pass
    return chunk_text


def truncate_to_max_tokens(text: str, max_tokens: int = 500) -> list[str]:
    """
    Pre-split oversized text into token-safe segments BEFORE passing to HybridChunker.
    Splits on newlines to preserve logical boundaries.
    """
    lines    = text.split("\n")
    segments = []
    current  = []
    current_tokens = 0

    for line in lines:
        line_tokens = len(tokenizer.encode(line, truncation=False))

        if current_tokens + line_tokens > max_tokens:
            if current:
                segments.append("\n".join(current))
            current        = [line]
            current_tokens = line_tokens
        else:
            current.append(line)
            current_tokens += line_tokens

    if current:
        segments.append("\n".join(current))

    return segments if segments else [text]


def chunk_page(page_text: str, page_num: int) -> list[dict]:
    
    # figures now come back as separate protected segments
    text, figure_segments, figures = inject_figures(page_text, page_num)

    results     = []
    chunk_index = 0

    # ── Chunk the main page text ──────────────────────────
    token_count_full = len(tokenizer.encode(text, truncation=False))
    segments = truncate_to_max_tokens(text) if token_count_full > 400 else [text]

    for segment in segments:
        doc    = build_doc(segment)
        chunks = list(chunker.chunk(doc))

        for chunk in chunks:
            chunk_text     = chunk.text
            token_count    = len(tokenizer.encode(chunk_text, truncation=True, max_length=512))
            heading        = extract_heading(chunk_text)
            contextualized = build_contextualized(chunk, chunk_text)

            results.append({
                "chunk_id":            f"page_{page_num}_chunk_{chunk_index}",
                "text":                chunk_text,
                "contextualized_text": contextualized,
                "page_no":             page_num,
                "heading":             heading,
                "figures_on_page":     figures,
                "token_count":         token_count,
            })
            chunk_index += 1

    # ── Add each figure description as its own atomic chunk ──
    for fig_text in figure_segments:
        token_count = len(tokenizer.encode(fig_text, truncation=True, max_length=512))

        # Skip if figure description itself is junk
        if len(fig_text.strip()) < MIN_CHUNK_CHARS:
            continue

        results.append({
            "chunk_id":            f"page_{page_num}_chunk_{chunk_index}",
            "text":                fig_text,
            "contextualized_text": fig_text,   # figure is self-contained, no heading needed
            "page_no":             page_num,
            "heading":             None,
            "figures_on_page":     figures,
            "token_count":         token_count,
            "is_figure_chunk":     True         # 🆕 flag for downstream filtering if needed
        })
        chunk_index += 1

    return results

In [69]:
#run full pipeline
all_chunks    = []
skipped_pages = 0
skipped_log   = []

for i, page in enumerate(tqdm(pages, desc="Chunking pages")):
    page_num = i + 1
    text = extract_text_from_doctags(page)

    if not text.strip():
        skipped_pages += 1
        skipped_log.append({
            "page_no": page_num,
            "reason":  "empty_text",
            "preview": repr(page[:100])
        })
        continue

    try:
        page_chunks = chunk_page(text, page_num)
    except Exception as e:
        print(f"[WARN] Failed to chunk page {page_num}: {e}")
        skipped_pages += 1
        skipped_log.append({
            "page_no": page_num,
            "reason":  str(e),
            "preview": repr(text[:100])
        })
        continue

    # Filter out tiny / junk chunks
    page_chunks = [c for c in page_chunks if len(c["text"].strip()) >= MIN_CHUNK_CHARS]
    all_chunks.extend(page_chunks)

print(f"\nTotal chunks   : {len(all_chunks)}")
print(f"Skipped pages  : {skipped_pages}")
print(f"Processed pages: {len(pages) - skipped_pages}")
print(f"Avg chunks/page: {len(all_chunks) / max(1, len(pages) - skipped_pages):.1f}")

Chunking pages: 100%|██████████| 654/654 [00:13<00:00, 47.99it/s] 


Total chunks   : 3055
Skipped pages  : 0
Processed pages: 654
Avg chunks/page: 4.7


In [70]:
#save output
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"Saved → {OUTPUT_PATH}")

Saved → ../data/chunks/enriched_chunks.json
